## 04c Mixed-Discard Supplementary Check

This notebook performs a **supplementary qualitative analysis** of the mixed-discard pairs that were excluded from the official Notebook 03 and Notebook 04 evaluation.

### Purpose

The course-level split used in Notebook 03 assigns a pair to:

- **dev** if both courses are in the development course set
- **test** if both courses are in the test course set
- **mixed_discard** if one course is in dev and the other is in test

Mixed-discard pairs were intentionally excluded from official benchmarking and threshold evaluation.

This notebook does **not** change the official results.

Instead, it asks a narrower question:

> How does the final selected model-threshold configuration behave on the excluded mixed-discard pairs?

### Why this notebook is useful

This supplementary check helps interpret the final model more honestly by showing:

- whether the selected configuration also identifies the discarded positive pairs
- whether it would produce many additional positive predictions among mixed-discard negatives
- whether its behavior on excluded pairs remains plausible and conservative

### Important methodological note

The mixed-discard pairs remain **out of scope for official model selection and threshold tuning**.

Results in this notebook should therefore be interpreted only as:

- a **qualitative robustness check**
- a **supplementary interpretation aid**
- not as part of the formal development/test benchmark

In [8]:
# Cell 01 - Import libraries and define project paths

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

BENCHMARK_PAIRS_DIR = DATA_DIR / "benchmark_pairs"
SPLITS_DIR = DATA_DIR / "splits"

THRESHOLD_TABLES_DIR = REPORTS_DIR / "tables" / "thresholds"
ERROR_ANALYSIS_TABLES_DIR = REPORTS_DIR / "tables" / "error_analysis"
MIXED_DISCARD_TABLES_DIR = REPORTS_DIR / "tables" / "mixed_discard_analysis"

PAIR_SCORES_CSV_PATH = BENCHMARK_PAIRS_DIR / "pair_similarity_scores_long.csv"
PAIR_SCORES_PARQUET_PATH = BENCHMARK_PAIRS_DIR / "pair_similarity_scores_long.parquet"

PAIRS_CSV_PATH = DATA_DIR / "gold_pairs_review_labeled.csv"
PAIR_SPLIT_CSV_PATH = SPLITS_DIR / "pair_dev_test_split_course_level.csv"
FINAL_RECOMMENDATION_CSV_PATH = THRESHOLD_TABLES_DIR / "final_threshold_recommendation.csv"

MIXED_DISCARD_FULL_CSV_PATH = MIXED_DISCARD_TABLES_DIR / "mixed_discard_full_selected_configuration.csv"
MIXED_DISCARD_POSITIVES_CSV_PATH = MIXED_DISCARD_TABLES_DIR / "mixed_discard_positives_selected_configuration.csv"
MIXED_DISCARD_SUMMARY_CSV_PATH = MIXED_DISCARD_TABLES_DIR / "mixed_discard_summary.csv"

MIXED_DISCARD_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Pair scores path:", PAIR_SCORES_PARQUET_PATH if PAIR_SCORES_PARQUET_PATH.exists() else PAIR_SCORES_CSV_PATH)
print("Pairs CSV:", PAIRS_CSV_PATH)
print("Pair split CSV:", PAIR_SPLIT_CSV_PATH)
print("Final recommendation CSV:", FINAL_RECOMMENDATION_CSV_PATH)
print("Output dir:", MIXED_DISCARD_TABLES_DIR)

Project root: C:\Users\user\Documents\thesis
Pair scores path: C:\Users\user\Documents\thesis\data\benchmark_pairs\pair_similarity_scores_long.parquet
Pairs CSV: C:\Users\user\Documents\thesis\data\gold_pairs_review_labeled.csv
Pair split CSV: C:\Users\user\Documents\thesis\data\splits\pair_dev_test_split_course_level.csv
Final recommendation CSV: C:\Users\user\Documents\thesis\reports\tables\thresholds\final_threshold_recommendation.csv
Output dir: C:\Users\user\Documents\thesis\reports\tables\mixed_discard_analysis


In [9]:
# Cell 02 - Load required data

import os
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

COURSES_CSV_PATH = DATA_DIR / "courses_clean.csv"
PAIRS_CSV_PATH = DATA_DIR / "gold_pairs_review_labeled.csv"
PAIR_SPLIT_CSV_PATH = SPLITS_DIR / "pair_dev_test_split_course_level.csv"
FINAL_RECOMMENDATION_CSV_PATH = THRESHOLD_TABLES_DIR / "final_threshold_recommendation.csv"
ENV_PATH = PROJECT_ROOT / ".env"

courses_df = pd.read_csv(COURSES_CSV_PATH, encoding="utf-8", keep_default_na=False)
pairs_df = pd.read_csv(PAIRS_CSV_PATH, encoding="utf-8", keep_default_na=False)
pair_split_df = pd.read_csv(PAIR_SPLIT_CSV_PATH, encoding="utf-8")
final_recommendation_df = pd.read_csv(FINAL_RECOMMENDATION_CSV_PATH, encoding="utf-8")

load_dotenv(ENV_PATH)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

print("courses_df shape:", courses_df.shape)
print("pairs_df shape:", pairs_df.shape)
print("pair_split_df shape:", pair_split_df.shape)
print("final_recommendation_df shape:", final_recommendation_df.shape)

display(final_recommendation_df)

courses_df shape: (50, 26)
pairs_df shape: (1225, 16)
pair_split_df shape: (1225, 4)
final_recommendation_df shape: (2, 17)


,recommendation_type,model_name,text_variant,similarity_metric,selection_rule,locked_threshold,shortlist_rank,average_precision,roc_auc,dev_f1,dev_f05,test_f1,test_f05,test_precision,test_recall,dev_predicted_positive_count,test_predicted_positive_count
0,primary_recommendation,openai_text-embedding-3-large,enriched,cosine,max_f1,0.673784,1,0.939595,0.994870,0.894737,0.923913,0.769231,0.892857,1.0,0.625,18,5
1,conservative_alternative,openai_text-embedding-3-large,main,cosine,max_f1,0.659979,2,0.927445,0.995913,0.871795,0.885417,0.769231,0.892857,1.0,0.625,19,5


In [10]:
# Cell 03 - Select the primary recommendation

primary_row = final_recommendation_df[
    final_recommendation_df["recommendation_type"] == "primary_recommendation"
].iloc[0]

selected_model_name = primary_row["model_name"]
selected_text_variant = primary_row["text_variant"]
selected_similarity_metric = primary_row["similarity_metric"]
selected_selection_rule = primary_row["selection_rule"]
selected_threshold = float(primary_row["locked_threshold"])

print("Selected primary recommendation:")
print("  model_name       :", selected_model_name)
print("  text_variant     :", selected_text_variant)
print("  similarity_metric:", selected_similarity_metric)
print("  selection_rule   :", selected_selection_rule)
print("  locked_threshold :", f"{selected_threshold:.6f}")

Selected primary recommendation:
  model_name       : openai_text-embedding-3-large
  text_variant     : enriched
  similarity_metric: cosine
  selection_rule   : max_f1
  locked_threshold : 0.673784


In [11]:
# Cell 04 - Attach split labels to all labeled pairs

pairs_with_split_df = pairs_df.merge(
    pair_split_df,
    on=["pair_id", "course_unit_code_1", "course_unit_code_2"],
    how="left"
)

if pairs_with_split_df["split"].isna().any():
    missing_count = int(pairs_with_split_df["split"].isna().sum())
    raise ValueError(f"Found {missing_count} rows with missing split labels after merge.")

print("Split distribution in labeled pairs:")
display(pairs_with_split_df["split"].value_counts())

mixed_discard_pairs_df = pairs_with_split_df[
    pairs_with_split_df["split"] == "mixed_discard"
].copy()

print("mixed_discard_pairs_df shape:", mixed_discard_pairs_df.shape)
display(
    mixed_discard_pairs_df[[
        "pair_id",
        "course_name_1",
        "course_name_2",
        "overlap_label",
        "review_notes",
    ]].head(10)
)

Split distribution in labeled pairs:


split
dev              595
mixed_discard    525
test             105
Name: count, dtype: int64

mixed_discard_pairs_df shape: (525, 17)


,pair_id,course_name_1,course_name_2,overlap_label,review_notes
2,TE00BR86__TE00CS89,Back-End Development,Databases,0,
7,TE00BR86__TT00CD76,Back-End Development,Scripting and Automatization,0,
12,TE00BR86__TT00CD81,Back-End Development,Databases,0,
16,TE00BR86__TT00CD86,Back-End Development,Cyber Security,0,
17,TE00BR86__TT00CD87,Back-End Development,Information Security Technologies,0,
18,TE00BR86__TT00CD88,Back-End Development,Hardening,0,
21,TE00BR86__TT00CD92,Back-End Development,Mobile Project,0,
25,TE00BR86__TT00CD98,Back-End Development,Mathematical Foundations of Machine Learning,0,
30,TE00BR86__TT00CE03,Back-End Development,Deep Learning,0,
32,TE00BR86__TT00CE06,Back-End Development,Machine Learning Project,0,


In [12]:
# Cell 05 - Helper functions for rebuilding scores for the selected configuration

def normalize_overlap_label(value):
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "overlap"}:
        return 1
    if text in {"0", "false", "no", "none", "no overlap"}:
        return 0
    return np.nan

def normalize_structured_text(text: str) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]
    return "\n".join(lines)

def prepare_text_for_model(text: str, model_name: str) -> str:
    text = normalize_structured_text(text)

    if model_name == "multilingual-e5-large-instruct":
        return (
            "Instruct: Retrieve semantically similar course descriptions for curriculum overlap detection\n"
            f"Query: {text}"
        )

    if model_name.startswith("multilingual-e5"):
        return f"query: {text}"

    return text

def l2_normalize_embeddings(matrix: np.ndarray) -> np.ndarray:
    return normalize(matrix, norm="l2")

def cosine_similarity_from_normalized(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return float(np.dot(vec1, vec2))

def dot_product_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return float(np.dot(vec1, vec2))

def batched(iterable, batch_size: int):
    for start in range(0, len(iterable), batch_size):
        yield iterable[start:start + batch_size]

In [13]:
# Cell 06 - Rebuild course embeddings for the selected final configuration

TEXT_VARIANTS = {
    "main": "embedding_text_main",
    "enriched": "embedding_text_enriched",
    "full": "embedding_text_full",
}

if selected_text_variant not in TEXT_VARIANTS:
    raise ValueError(f"Unsupported selected_text_variant: {selected_text_variant}")

text_column = TEXT_VARIANTS[selected_text_variant]

course_text_df = courses_df[[
    "course_unit_code",
    "course_name",
    text_column,
]].copy()

course_text_df[text_column] = course_text_df[text_column].astype(str)
prepared_texts = [
    prepare_text_for_model(text, model_name=selected_model_name)
    for text in course_text_df[text_column].tolist()
]

openai_client = None
raw_embedding_matrix = None
normalized_embedding_matrix = None

if selected_model_name == "openai_text-embedding-3-large":
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY not found in .env, but the selected final model is OpenAI.")

    openai_client = OpenAI(api_key=OPENAI_API_KEY)

    all_vectors = []
    for batch_texts in batched(prepared_texts, batch_size=64):
        response = openai_client.embeddings.create(
            model="text-embedding-3-large",
            input=batch_texts,
        )
        batch_vectors = [item.embedding for item in response.data]
        all_vectors.extend(batch_vectors)

    raw_embedding_matrix = np.array(all_vectors, dtype=np.float32)
    normalized_embedding_matrix = l2_normalize_embeddings(raw_embedding_matrix)

elif selected_model_name in {
    "multilingual-e5-base",
    "multilingual-e5-large",
    "multilingual-e5-large-instruct",
    "bge-m3",
}:
    hf_model_map = {
        "multilingual-e5-base": "intfloat/multilingual-e5-base",
        "multilingual-e5-large": "intfloat/multilingual-e5-large",
        "multilingual-e5-large-instruct": "intfloat/multilingual-e5-large-instruct",
        "bge-m3": "BAAI/bge-m3",
    }

    st_model = SentenceTransformer(hf_model_map[selected_model_name])
    raw_embedding_matrix = st_model.encode(
        prepared_texts,
        batch_size=8,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32)
    normalized_embedding_matrix = l2_normalize_embeddings(raw_embedding_matrix)

else:
    raise ValueError(
        f"Notebook 04c currently supports dense selected models only. Got: {selected_model_name}"
    )

print("raw_embedding_matrix shape:", raw_embedding_matrix.shape)
print("normalized_embedding_matrix shape:", normalized_embedding_matrix.shape)

raw_embedding_matrix shape: (50, 3072)
normalized_embedding_matrix shape: (50, 3072)


In [14]:
# Cell 07 - Score all labeled pairs with the selected final configuration

pairs_with_split_df["human_binary"] = pairs_with_split_df["overlap_label"].apply(normalize_overlap_label)

if pairs_with_split_df["human_binary"].isna().any():
    bad_rows = pairs_with_split_df.loc[
        pairs_with_split_df["human_binary"].isna(),
        ["pair_id", "overlap_label"]
    ]
    raise ValueError(f"Found invalid overlap labels.\n{bad_rows}")

course_code_order = course_text_df["course_unit_code"].tolist()
code_to_index = {code: idx for idx, code in enumerate(course_code_order)}

all_selected_score_rows = []

for _, row in pairs_with_split_df.iterrows():
    idx1 = code_to_index[row["course_unit_code_1"]]
    idx2 = code_to_index[row["course_unit_code_2"]]

    if selected_similarity_metric == "cosine":
        vec1 = normalized_embedding_matrix[idx1]
        vec2 = normalized_embedding_matrix[idx2]
        similarity_score = cosine_similarity_from_normalized(vec1, vec2)
    elif selected_similarity_metric == "dot":
        vec1 = raw_embedding_matrix[idx1]
        vec2 = raw_embedding_matrix[idx2]
        similarity_score = dot_product_similarity(vec1, vec2)
    else:
        raise ValueError(f"Unsupported similarity metric for 04c: {selected_similarity_metric}")

    ai_binary = int(similarity_score >= selected_threshold)

    if row["human_binary"] == ai_binary:
        error_type = "correct"
    elif row["human_binary"] == 1 and ai_binary == 0:
        error_type = "false_negative"
    else:
        error_type = "false_positive"

    all_selected_score_rows.append({
        "pair_id": row["pair_id"],
        "split": row["split"],
        "course_unit_code_1": row["course_unit_code_1"],
        "course_name_1": row["course_name_1"],
        "course_unit_code_2": row["course_unit_code_2"],
        "course_name_2": row["course_name_2"],
        "review_notes": row["review_notes"],
        "human_binary": int(row["human_binary"]),
        "ai_binary": ai_binary,
        "similarity_score": float(similarity_score),
        "error_type": error_type,
    })

all_selected_scores_df = pd.DataFrame(all_selected_score_rows)

print("all_selected_scores_df shape:", all_selected_scores_df.shape)
display(all_selected_scores_df.head(10))

all_selected_scores_df shape: (1225, 11)


,pair_id,split,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,review_notes,human_binary,ai_binary,similarity_score,error_type
0,TE00BR86__TE00CN56,dev,TE00BR86,Back-End Development,TE00CN56,Internet Networks and Security,,0,0,0.460944,correct
1,TE00BR86__TE00CS88,dev,TE00BR86,Back-End Development,TE00CS88,Introduction to Programming,,0,0,0.501618,correct
2,TE00BR86__TE00CS89,mixed_discard,TE00BR86,Back-End Development,TE00CS89,Databases,,0,0,0.549394,correct
3,TE00BR86__TT00CD70,dev,TE00BR86,Back-End Development,TT00CD70,Data Networks,,0,0,0.442442,correct
4,TE00BR86__TT00CD71,dev,TE00BR86,Back-End Development,TT00CD71,Linux Basics,,0,0,0.432360,correct
5,TE00BR86__TT00CD72,dev,TE00BR86,Back-End Development,TT00CD72,Servers and containers,,0,0,0.488771,correct
6,TE00BR86__TT00CD75,dev,TE00BR86,Back-End Development,TT00CD75,Windows Basics,,0,0,0.394789,correct
7,TE00BR86__TT00CD76,mixed_discard,TE00BR86,Back-End Development,TT00CD76,Scripting and Automatization,,0,0,0.426708,correct
8,TE00BR86__TT00CD77,dev,TE00BR86,Back-End Development,TT00CD77,Basics of Programming,,0,0,0.506899,correct
9,TE00BR86__TT00CD78,dev,TE00BR86,Back-End Development,TT00CD78,Frontend Programming,,0,0,0.615649,correct


In [15]:
# Cell 08 - Keep only mixed-discard pairs for supplementary analysis

mixed_discard_eval_df = all_selected_scores_df[
    all_selected_scores_df["split"] == "mixed_discard"
].copy()

mixed_discard_eval_df.to_csv(MIXED_DISCARD_FULL_CSV_PATH, index=False, encoding="utf-8")

print("Saved mixed-discard full table to:", MIXED_DISCARD_FULL_CSV_PATH)
print("mixed_discard_eval_df shape:", mixed_discard_eval_df.shape)

display(
    mixed_discard_eval_df[[
        "pair_id",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "error_type",
        "review_notes",
    ]].head(15)
)

Saved mixed-discard full table to: C:\Users\user\Documents\thesis\reports\tables\mixed_discard_analysis\mixed_discard_full_selected_configuration.csv
mixed_discard_eval_df shape: (525, 11)


,pair_id,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,error_type,review_notes
2,TE00BR86__TE00CS89,Back-End Development,Databases,0.549394,0,0,correct,
7,TE00BR86__TT00CD76,Back-End Development,Scripting and Automatization,0.426708,0,0,correct,
12,TE00BR86__TT00CD81,Back-End Development,Databases,0.469883,0,0,correct,
16,TE00BR86__TT00CD86,Back-End Development,Cyber Security,0.453193,0,0,correct,
17,TE00BR86__TT00CD87,Back-End Development,Information Security Technologies,0.470954,0,0,correct,
18,TE00BR86__TT00CD88,Back-End Development,Hardening,0.480115,0,0,correct,
21,TE00BR86__TT00CD92,Back-End Development,Mobile Project,0.515647,0,0,correct,
25,TE00BR86__TT00CD98,Back-End Development,Mathematical Foundations of Machine Learning,0.318678,0,0,correct,
30,TE00BR86__TT00CE03,Back-End Development,Deep Learning,0.434371,0,0,correct,
32,TE00BR86__TT00CE06,Back-End Development,Machine Learning Project,0.412469,0,0,correct,


In [16]:
# Cell 09 - Summary of mixed-discard behavior

mixed_discard_summary_df = pd.DataFrame([{
    "mixed_discard_pair_count": int(len(mixed_discard_eval_df)),
    "mixed_discard_positive_count": int((mixed_discard_eval_df["human_binary"] == 1).sum()),
    "mixed_discard_negative_count": int((mixed_discard_eval_df["human_binary"] == 0).sum()),
    "ai_predicted_positive_count": int((mixed_discard_eval_df["ai_binary"] == 1).sum()),
    "ai_predicted_negative_count": int((mixed_discard_eval_df["ai_binary"] == 0).sum()),
    "true_positive_count": int(((mixed_discard_eval_df["human_binary"] == 1) & (mixed_discard_eval_df["ai_binary"] == 1)).sum()),
    "true_negative_count": int(((mixed_discard_eval_df["human_binary"] == 0) & (mixed_discard_eval_df["ai_binary"] == 0)).sum()),
    "false_positive_count": int(((mixed_discard_eval_df["human_binary"] == 0) & (mixed_discard_eval_df["ai_binary"] == 1)).sum()),
    "false_negative_count": int(((mixed_discard_eval_df["human_binary"] == 1) & (mixed_discard_eval_df["ai_binary"] == 0)).sum()),
}])

mixed_discard_summary_df.to_csv(MIXED_DISCARD_SUMMARY_CSV_PATH, index=False, encoding="utf-8")

print("Saved mixed-discard summary to:", MIXED_DISCARD_SUMMARY_CSV_PATH)
display(mixed_discard_summary_df)

Saved mixed-discard summary to: C:\Users\user\Documents\thesis\reports\tables\mixed_discard_analysis\mixed_discard_summary.csv


,mixed_discard_pair_count,mixed_discard_positive_count,mixed_discard_negative_count,ai_predicted_positive_count,ai_predicted_negative_count,true_positive_count,true_negative_count,false_positive_count,false_negative_count
0,525,4,521,8,517,4,517,4,0


In [17]:
# Cell 10 - Inspect the discarded positive pairs specifically

mixed_discard_positives_df = mixed_discard_eval_df[
    mixed_discard_eval_df["human_binary"] == 1
].copy()

mixed_discard_positives_df = mixed_discard_positives_df.sort_values(
    "similarity_score",
    ascending=False
).reset_index(drop=True)

mixed_discard_positives_df.to_csv(
    MIXED_DISCARD_POSITIVES_CSV_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved discarded positives table to:", MIXED_DISCARD_POSITIVES_CSV_PATH)
print("Discarded positive count:", len(mixed_discard_positives_df))

display(
    mixed_discard_positives_df[[
        "pair_id",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "error_type",
        "review_notes",
    ]]
)

Saved discarded positives table to: C:\Users\user\Documents\thesis\reports\tables\mixed_discard_analysis\mixed_discard_positives_selected_configuration.csv
Discarded positive count: 4


,pair_id,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,error_type,review_notes
0,TT00CE05__TT00CE06,Data Analytics Project,Machine Learning Project,0.778475,1,1,correct,"Both are AI/DA related projects, same content exists, but not fully overlap"
1,TT00CD70__TT00CE28,Data Networks,Core Networks,0.743941,1,1,correct,"Both cover routing protocols (OSPF, BGP), network design, and WAN connectivity as core content."
2,TT00CE09__TT00CE17,Auditing and Penetration Testing,Web Application Security Testing,0.692866,1,1,correct,"Both cover offensive security testing, vulnerability discovery, and exploitation-oriented assessment work."
3,TT00CE20__TT00CE28,Local Area Networks,Core Networks,0.690693,1,1,correct,"Both cover backbone and LAN networking protocols, routing, and network design as core content."


In [18]:
# Cell 11 - Inspect mixed-discard false positives

mixed_discard_false_positive_df = mixed_discard_eval_df[
    mixed_discard_eval_df["error_type"] == "false_positive"
].copy()

MIXED_DISCARD_FALSE_POSITIVES_CSV_PATH = (
    MIXED_DISCARD_TABLES_DIR / "mixed_discard_false_positives_selected_configuration.csv"
)
mixed_discard_false_positive_df.to_csv(
    MIXED_DISCARD_FALSE_POSITIVES_CSV_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved mixed-discard false positives to:", MIXED_DISCARD_FALSE_POSITIVES_CSV_PATH)
print("Mixed-discard false positives:", len(mixed_discard_false_positive_df))

display(
    mixed_discard_false_positive_df[[
        "pair_id",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "error_type",
        "review_notes",
    ]].sort_values("similarity_score", ascending=False).reset_index(drop=True)
)

Saved mixed-discard false positives to: C:\Users\user\Documents\thesis\reports\tables\mixed_discard_analysis\mixed_discard_false_positives_selected_configuration.csv
Mixed-discard false positives: 4


,pair_id,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,error_type,review_notes
0,TT00CD92__TT00CD94,Mobile Project,Mobile Application Development,0.830978,0,1,false_positive,Mobile project course versus mobile application development course; project application rather than substantial content redundancy.
1,TT00CD92__TT00DS75,Mobile Project,Mobile Application Development,0.787621,0,1,false_positive,"Mobile project course versus mobile development course; adjacent and practical, but not enough repeated teaching content."
2,TT00CD92__TT00CD93,Mobile Project,Android Application Development,0.688586,0,1,false_positive,Mobile project course versus Android implementation course; strong domain relation but not enough repeated core curriculum.
3,TE00CN56__TT00CD87,Internet Networks and Security,Information Security Technologies,0.683702,0,1,false_positive,


### Reading of the mixed-discard false positives

The four mixed-discard false positives are semantically plausible rather than arbitrary.

Three of them involve **Mobile Project** paired with mobile-development courses. In these cases, the model appears to respond to strong domain similarity and shared practical context. However, the expert labeling criterion is stricter: a project course in the same domain is not automatically treated as substantial curriculum overlap unless the repeated taught content is clearly strong enough.

The remaining false positive, **Internet Networks and Security** versus **Information Security Technologies**, shows a similar pattern. The pair shares clear technical-security themes, but the expert judgment draws a narrower boundary between thematic relatedness and true curriculum overlap.

This strengthens the interpretation that the selected configuration mainly over-predicts in **semantically adjacent** cases, not in obviously unrelated ones. That is an important distinction, because it suggests the model is capturing meaningful similarity while still occasionally crossing the stricter expert threshold for what should count as overlap.

## Interpretation of mixed-discard supplementary analysis

This supplementary check was not part of the main benchmark or threshold-selection pipeline.  
Its purpose was only to inspect how the selected final configuration behaves on the course pairs that were excluded from the formal evaluation because they crossed the course-level development/test boundary.

The result is encouraging.

Among the **4 expert-labeled positive mixed-discard pairs**, the selected final configuration correctly identified **all 4 as overlaps**. This suggests that the excluded positives were not unusually difficult cases for the model and that the selected threshold still captures them credibly.

At the same time, the model predicted **8 positives** among the **525 mixed-discard pairs**, meaning that it also produced **4 false positives** in this supplementary set. This is consistent with the broader thesis finding that semantic similarity can sometimes surface thematically related courses even when the expert judgment does not consider the overlap strong enough for a positive label.

This analysis does **not** change the formal Notebook 03 or Notebook 04 results, because mixed-discard pairs were excluded intentionally to preserve the integrity of the course-level split.  
However, it adds useful supporting evidence that the selected final configuration behaves reasonably even outside the strictly retained evaluation subset.